## Params recommended
  *   Word embedding: (One-hot encoding,FastText), (GloVe(300), Word2Vec is not for char based)
  *   Learning rate (alpha): 0.001
  *   Dropout: 0.2 ~ 0.5
  *   Epoch for word learning: 5 ~ 50
  *   Epoch for dataset: 10 ~ 100
  *   Activation function: ReLU (Rectified Linear Unit), Sigmoid
  *   Error function (Loss function):
    *   Mean Squared Error (MSE)
    *   Cross Entropy Loss
    *   Binary Cross-Entropy
  *   Type of GD (Gradient Descent):
    *   Stochastic (batch=1),
    *   Batch
    *   Mini-batch (batch_size=32, 64):
  *   Optimizers:
    *   Adam
    *   RMSProp
  *   Model:
    *   LSTM,
    *   BiLSTM
    *   Transformer encoder (because order of characters matters)


## Used params
*   Word embedding: One-hot encoding
*   Learning rate (alpha): 0.001
*   Dropout: 0.2 ~ 0.5
*   Epoch for word learning: 5 ~ 50
*   Epoch for dataset: 10 ~ 100
*   Activation function: ReLU (Rectified Linear Unit), Sigmoid
*   Error function (Loss function):
   *   Mean Squared Error (MSE)
   *   Cross Entropy Loss
   *   Binary Cross-Entropy
*   Type of GD (Gradient Descent):
   *   Stochastic (batch=1),
   *   Batch
   *   Mini-batch (batch_size=32, 64):
*   Optimizers:
   *   Adam
   *   RMSProp
*   Model:
	*   LSTM,
  *   BiLSTM
  *   Transformer encoder (because order of characters matters)



### Neural Net Architecture:
- Input layer size  = char_len × max_word_len
- Output layer size = same as input (autoencoder-like structure)
- Activation        = Sigmoid
- Loss              = Mean Squared Error (MSE)

Trainable Parameters:
- weight: shape = [input_dim, output_dim], random init
- bias: shape = [output_dim], random init

Training Hyperparameters:
- l_rate (learning rate)    → user-defined
- max_epoch (per word)      → user-defined
- error (MSE threshold)     → user-defined


In [ ]:
# ============================
# 1. Install deps
# ============================
!pip install -U evaluate transformers datasets sentencepiece sacrebleu

import os
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
import evaluate
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)


# ============================
# 2. Load dataset
# ============================
from google.colab import drive
drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/Colab Notebooks/Prog/TrainData.txt"

surface_words, stems = [], []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        if "/" not in line: continue
        parts = line.split("/")
        stem, ending = parts[0], parts[1] if len(parts) > 1 else ""
        surface = stem + ending
        surface_words.append(surface)
        stems.append(stem)

df = pd.DataFrame({"input": surface_words, "target": stems})
print("Sample data:\n", df.head())

# Train-test split
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
datasets = DatasetDict({"train": train_dataset, "test": test_dataset})

# ============================
# 3. Model & Tokenizer
# ============================
# You can also try "google/mt5-small" for multilingual support
model_name = "t5-base"
# model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ============================
# 4. Preprocessing
# ============================
max_input_length = 32
max_target_length = 32

def preprocess_function(examples):
    inputs = examples["input"]
    targets = examples["target"]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)


    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_datasets = datasets.map(preprocess_function, batched=True)

# ============================
# 5. Model & Collator
# ============================
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

# ============================
# 6. Metrics
# ============================
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

# def compute_metrics(eval_preds):
#     preds, labels = eval_preds
#     if isinstance(preds, tuple):
#         preds = preds[0]
#     decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

#     decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
#     result = metric.compute(predictions=decoded_preds, references=decoded_labels)
#     return {"bleu": result["score"]}


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 with pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    # Use accuracy instead of BLEU for stemming
    correct = sum(p == l[0] for p, l in zip(decoded_preds, decoded_labels))
    acc = correct / len(decoded_preds)

    return {"accuracy": acc}

# ============================
# 7. TrainingArguments
# ============================
batch_size = 16
args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=20,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",   # 🔥 change this line
    greater_is_better=True,
)

# ============================
# 8. Trainer
# ============================
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],

    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

# ============================
# 9. Train & Evaluate
# ============================
train_result = trainer.train()
eval_results = trainer.evaluate()
print("Evaluation:", eval_results)

# ============================
# 10. Inference (Stemming)
# ============================
def stem_word(word):
    inputs = tokenizer(word, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=32)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test
test_words = ["atrofidagi", "auksionga", "kitoblarni"]
for w in test_words:
    print(w, "->", stem_word(w))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.1 MB/s eta 0:00:00
Mounted at /content/drive
Sample data:
         input      target
0       a’lam       a’lam
1        a’lo        a’lo
2     a’lochi        a’lo
3  a’lochilik        a’lo
4  a’lohazrat  a’lohazrat


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ulugbek0302 (ulugbek0302-urgench-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,0.466000,0.327217,0.652218
2,0.327000,0.251946,0.716734
3,0.225400,0.231752,0.759073
4,0.193200,0.221844,0.780242
5,0.160600,0.197326,0.813508
6,0.114100,0.199688,0.830645
7,0.097900,0.213463,0.829637
8,0.063800,0.209075,0.833669
9,0.067000,0.210286,0.840726
10,0.055300,0.222005,0.837702


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Evaluation: {'eval_loss': 0.22827376425266266, 'eval_accuracy': 0.8578629032258065, 'eval_runtime': 12.6914, 'eval_samples_per_second': 78.163, 'eval_steps_per_second': 4.885, 'epoch': 20.0}
atrofidagi -> atrof
auksionga -> auksion
kitoblarni -> kitob
